# Tests : calcul des coordonnées des coins SG et ID d'un champ récepteur d'activation

Created : *19/12/2024*

In [1]:
import torch
import torchvision.models as models

In [2]:
from utils import get_receptive_field_2d, get_receptive_field_in_pixel_space, get_output_sizes

## Calcul du champ réceptif d'une activation de la sortie d'un module au sein de son espace d'entrée.

In [3]:
tests = [
    {   #0
        "pos": (0, 0),
        "kernel": (3, 3),
        "stride": 1,
        "padding": 0,
        "input_size": None,
        "expected": ((0, 0), (2, 2))
    },
    {   #1
        "pos": (0, 0),
        "kernel": (3,3),
        "stride": 1,
        "padding": 1,
        "input_size": None,
        "expected": ((-1, -1), (1, 1))
    },
    {
        "pos": (0, 0),
        "kernel": (3, 3),
        "stride": 1,
        "padding": 1,
        "input_size": (6, 4),
        "expected": ((0, 0), (1, 1))
    },
    {
        "pos": (0, 1),
        "kernel": (3, 3),
        "stride": 1,
        "padding": 0,
        "input_size": (6, 4),
        "expected": ((0, 1), (2, 3))
    },
    {
        "pos": (0, 1),
        "kernel": (3, 3),
        "stride": 2,
        "padding": 0,
        "input_size": (6, 4),
        "expected": ((0, 2), (2, 4))
    },
    {   #5
        "pos": (1, 1),
        "kernel": (3, 3),
        "stride": 2,
        "padding": 1,
        "input_size": (6, 4),
        "expected": ((1, 1), (3, 3))
    },
    {   
        "pos": (2, 2),
        "kernel": (3, 3),
        "stride": 2,
        "padding": 1,
        "input_size": (6, 4),
        "expected": ((3, 3), (3, 5))
    },
]

for i, test in enumerate(tests):
    current = get_receptive_field_2d(
        pos=test["pos"],
        kernel=test["kernel"],
        stride=test["stride"],
        padding=test["padding"],
        input_size=test["input_size"],
    )
    assert current == test["expected"], f"Test #{i} : expected {test['expected']} instead of {current}"
    print("ok :", current)

ok : ((0, 0), (2, 2))
ok : ((-1, -1), (1, 1))
ok : ((0, 0), (1, 1))
ok : ((0, 1), (2, 3))
ok : ((0, 2), (2, 4))
ok : ((1, 1), (3, 3))
ok : ((3, 3), (3, 5))


## Calcul du champ réceptif dans l'espace des pixels, d'une activation d'une des couches d'un modèle de type CNN

In [4]:
model = models.alexnet(weights='IMAGENET1K_V1')
model.eval()
model.features

Sequential(
  (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
  (1): ReLU(inplace=True)
  (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (4): ReLU(inplace=True)
  (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU(inplace=True)
  (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): ReLU(inplace=True)
  (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
)

In [7]:
input_size=torch.Size([1, 3, 224, 224])
output_sizes = get_output_sizes(model.features, input_size=input_size)
print(output_sizes)

[(55, 55), (55, 55), (27, 27), (27, 27), (27, 27), (13, 13), (13, 13), (13, 13), (13, 13), (13, 13), (13, 13), (13, 13), (6, 6)]


In [6]:
#idx_layer = 0 # 1st Conv2d output
idx_layer = 7 # 3rd ReLU output
idx_layer = -1 # last module output

pixel_space_size = (input_size[-2], input_size[-1])
receptive_field = get_receptive_field_in_pixel_space(
    pos=(3, 3),
    idx_layer=idx_layer, # 3rd ReLU output
    cnn_modules=model.features,
    output_sizes=output_sizes,
    pixel_space_size=pixel_space_size
)
print(receptive_field)

12 MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
11 ReLU(inplace=True)
10 Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
9 ReLU(inplace=True)
8 Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
7 ReLU(inplace=True)
6 Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
5 MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
4 ReLU(inplace=True)
3 Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
2 MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
1 ReLU(inplace=True)
0 Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
((30, 30), (223, 223))
